# Entrenamiento XGBoost — Iris y MLflow 3

Este notebook entrena, evalúa y registra una versión `Challenger` de `iris_classifier`. Mantiene visibles las llamadas nativas a XGBoost y MLflow para que el flujo sea auditable desde el propio notebook.

**Fuentes por runtime**

- Databricks: `workspace.default.iris_features` mediante Spark/Delta y registro en Unity Catalog.
- Local: `data/local/iris_features.csv` y MLflow con SQLite.

**Resultado esperado:** un run con parámetros, métricas y artefactos de evaluación; una nueva versión del modelo con tags y descripción; y el alias `Challenger` apuntando a esa versión. Este notebook nunca asigna `Champion` ni despliega Model Serving.

## 1. Preparar dependencias

En Databricks se instala el paquete local desde la raíz del repositorio y el extra oficial de MLflow; luego se reinicia Python para activar el entorno. En ejecución local esta celda no hace nada porque las dependencias ya provienen de `.venv`. Después del reinicio en Databricks, continúa desde la siguiente celda.

In [ ]:
if 'dbutils' in globals():
    get_ipython().run_line_magic('pip', 'install ../.. "mlflow[databricks]>=3.1,<4"')
    dbutils.library.restartPython()

## 2. Resolver runtime y configuración

La celda detecta `local` o `databricks`, carga `config/training.toml`, selecciona la fuente de datos y configura Tracking/Registry. Las variables de entorno sólo sobrescriben valores permitidos. Revisa las líneas `Runtime`, `Dataset`, `Registry` y `Model` antes de entrenar: son el contrato efectivo de esta ejecución.

Las utilidades se limitan a configuración, validación de datos, evaluación común y metadatos. Las operaciones de entrenamiento y MLflow permanecen explícitas mediante APIs nativas.

In [ ]:
# La configuración versionada se carga desde config/training.toml.
import json
from dataclasses import replace
import logging
import pandas as pd
import tempfile
import time

import matplotlib.pyplot as plt
import mlflow
import mlflow.pyfunc
import mlflow.xgboost
from mlflow import MlflowClient
from mlflow.exceptions import MlflowException
from mlflow.models import evaluate, infer_signature
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

from iris_mlflow_utils import (
    build_classification_table,
    build_config,
    build_metrics_summary_table,
    build_probability_metrics,
    build_registry_client,
    ensure_mlflow_experiment,
    evaluate_train_test,
    detect_runtime,
    load_dataset_for_runtime,
    get_delta_table_version,
    synchronize_model_registry_metadata,
)

MODEL_KEY = "xgboost"  # Clave estable de la sección TOML.
RUNTIME_MODE = detect_runtime()
config = build_config(
    model_slug=MODEL_KEY,
)
MODEL_TYPE = config.model_type
MODEL_FRAMEWORK = config.model_framework
PIP_REQUIREMENTS = ["mlflow", "numpy", "pandas", "xgboost"]
CONDA_ENV = {"channels": ["conda-forge"], "dependencies": ["python=3.12", {"pip": PIP_REQUIREMENTS}]}
model_params = {**config.model_params, "random_state": config.random_state}
if config.tracking_uri:
    mlflow.set_tracking_uri(config.tracking_uri)  # Backend explícito solo si fue configurado.
mlflow.set_registry_uri(config.registry_uri)  # Registry UC o URI parametrizada.
ensure_mlflow_experiment(config.experiment_name, tracking_uri=config.tracking_uri, artifact_location=config.artifact_location)  # Crea o restaura el experimento.
if not config.enable_tracing:
    mlflow.tracing.disable()
print(f"Runtime: {RUNTIME_MODE}")
print(f"Experimento: {config.experiment_name}")
FEATURE_TABLE = str(config.dataset_path) if RUNTIME_MODE == 'local' else config.feature_table
FEATURE_TABLE_VERSION = 'local' if RUNTIME_MODE == 'local' else (config.feature_table_version or get_delta_table_version(globals().get('spark'), config.feature_table))
config = replace(config, feature_table_version=FEATURE_TABLE_VERSION)
print(f"Modelo: {config.registered_model_name}")
print(f"Feature table: {FEATURE_TABLE}")

## 3. Cargar datos y construir el split

Se valida el contrato `Id + 4 features + Species`, pero `Id` no entra al modelo. Las cuatro features se convierten a `float64`; `train_test_split` conserva la proporción de las tres especies mediante `stratify`. Los spans de MLflow guardan únicamente metadatos resumidos, nunca el dataset completo.

In [ ]:
with mlflow.start_span(name="iris.load_dataset", span_type="CHAIN", attributes={"dataset_version": config.dataset_version, "feature_table": FEATURE_TABLE}) as span:
    dataset = load_dataset_for_runtime(
        runtime_mode=RUNTIME_MODE,
        spark=globals().get('spark'),
        config=config,
        table_version=None if RUNTIME_MODE == 'local' else FEATURE_TABLE_VERSION,
    )
    feature_table_exists = True
    feature_table_created = False
    span.set_outputs({"rows": len(dataset.dataframe), "features": len(dataset.feature_columns), "feature_table_version": config.feature_table_version or "latest"})

with mlflow.start_span(name="iris.split_dataset", span_type="CHAIN", attributes={"random_state": config.random_state, "test_size": config.test_size}) as span:
    x_train, x_test, y_train, y_test = train_test_split(
        dataset.features,  # Variables predictoras sin Id.
        dataset.target,  # Clases codificadas para estratificar.
        test_size=config.test_size,  # Proporción reservada para test.
        random_state=config.random_state,  # Semilla reproducible.
        stratify=dataset.target,  # Conserva la proporción de clases.
    )
    x_train = x_train.astype("float64")
    x_test = x_test.astype("float64")
    span.set_outputs({"train_rows": len(x_train), "test_rows": len(x_test)})
print(f"Train: {len(x_train)} | Test: {len(x_test)} | Clases: {list(dataset.classes)}")
display(dataset.dataframe.head())

## 4. Entrenar, evaluar y registrar

Dentro de un único `mlflow.start_run` se registran parámetros reproducibles, métricas homologadas y artefactos clásicos. El modelo se publica con `mlflow.xgboost.log_model`, firma de cuatro columnas, ejemplo de entrada y dependencias explícitas. Después se agregan tags, comentarios y el alias `Challenger`.

APIs nativas principales: `XGBClassifier.fit`, `mlflow.log_params`, `mlflow.log_metrics`, `mlflow.models.evaluate`, `mlflow.log_table` y `mlflow.xgboost.log_model`. La URI válida siempre se obtiene de `model_info.model_uri`.

In [ ]:
model = XGBClassifier(num_class=len(dataset.classes), **model_params)
with mlflow.start_run(
    run_name=config.run_name,  # Nombre descriptivo del entrenamiento.
) as run:
    run_id = run.info.run_id
    mlflow.log_params(
        {**model_params, "test_size": config.test_size, "random_state": config.random_state, "dataset_version": config.dataset_version, "feature_columns": json.dumps(dataset.feature_columns), "feature_table": FEATURE_TABLE, "feature_table_version": config.feature_table_version or "latest", "feature_table_exists": str(feature_table_exists).lower(), "feature_table_created": str(feature_table_created).lower()},  # Parámetros reproducibles del entrenamiento.
    )
    mlflow.set_tags(
        {"model_type": MODEL_TYPE, "stage": "Challenger", "algorithm": type(model).__name__, "framework": MODEL_FRAMEWORK, "training_type": "baseline", "dataset": "iris", "dataset_version": config.dataset_version, "project_version": config.project_version, "author": config.author, "purpose": config.purpose, "primary_metric": config.primary_metric, "tracking_backend": "databricks-managed" if not config.tracking_uri else config.tracking_uri, "feature_table_source": "local_file" if RUNTIME_MODE == "local" else "unity_catalog", "feature_table_version": FEATURE_TABLE_VERSION, "evaluation_status": "started"},  # Tags de trazabilidad y estado del modelo.
    )
    with mlflow.start_span(
        name="iris.train.XGBoost",  # Nombre visible de la etapa de entrenamiento.
        span_type="CHAIN",  # Clasifica el span como una cadena de procesamiento.
        attributes={"model_type": type(model).__name__, "train_rows": len(x_train)},  # Metadatos técnicos sin datos sensibles.
    ) as span:
        model.fit(x_train, y_train)
        span.set_outputs({"status": "completed"})
    evaluations = evaluate_train_test(model, x_train, x_test, y_train, y_test, len(dataset.classes))
    metrics = {f"{partition}_{name}": value for partition, result in evaluations.items() for name, value in result.metrics.items()}
    for partition, target in (("train", y_train), ("test", y_test)):
        metrics.update({f"{partition}_{name}": value for name, value in build_probability_metrics(evaluations[partition], target, list(range(len(dataset.classes)))).items()})
    mlflow.log_metrics(metrics=metrics)  # Métricas train/test del modelo.
    mlflow.log_dict(evaluations["test"].report, "evaluation/classification_report.json")  # Reporte de clasificación en JSON.
    mlflow.log_dict({str(i): name for i, name in enumerate(dataset.classes)}, "class_mapping.json")  # Mapping de clases para serving.
    metrics_summary = build_metrics_summary_table(evaluations, model_type=MODEL_TYPE, dataset_version=config.dataset_version, project_version=config.project_version, run_id=run_id)
    classification_by_class = build_classification_table(evaluations, dataset.classes, model_type=MODEL_TYPE, dataset_version=config.dataset_version, project_version=config.project_version, run_id=run_id)
    mlflow.log_table(data=metrics_summary, artifact_file="evaluation/metrics_summary.json")  # Tabla agregada de métricas.
    mlflow.log_table(data=classification_by_class, artifact_file="evaluation/classification_by_class.json")  # Tabla por clase.
    figure, axis = plt.subplots(figsize=(6, 5))  # Figura local que se guardará como artefacto.
    axis.imshow(evaluations["test"].confusion_matrix, cmap="Blues")
    axis.set(xticks=range(len(dataset.classes)), yticks=range(len(dataset.classes)), xticklabels=dataset.classes, yticklabels=dataset.classes, xlabel="Predicción", ylabel="Valor real", title="Matriz de confusión")
    figure.tight_layout()
    with tempfile.TemporaryDirectory() as directory:
        path = f"{directory}/confusion_matrix.png"
        figure.savefig(path, dpi=150)
        mlflow.log_artifact(path, artifact_path="evaluation")  # Artefacto visual asociado al run.
    plt.close(figure)
    signature = infer_signature(x_train, model.predict(x_train))  # Firma inferida del contrato real del modelo.
    with mlflow.start_span(name="iris.log_model", span_type="CHAIN", attributes={"model_type": MODEL_TYPE}) as span:
        model_info = mlflow.xgboost.log_model(
            xgb_model=model,  # Estimador XGBoost ajustado que se serializará.
            name="model",  # Nombre del Logged Model dentro del run.
            signature=signature,  # Contrato de columnas y tipos de entrada/salida.
            input_example=x_train.head(config.model_input_example_rows),  # Ejemplo configurable para documentar la firma.
            conda_env=CONDA_ENV,
            model_type="classifier",
        )
        span.set_outputs({"model_uri": model_info.model_uri, "model_id": model_info.model_id})
    evaluation_data = x_test.copy()
    evaluation_data[config.target_column] = pd.Categorical(y_test, categories=list(range(len(dataset.classes))))  # Declara las clases para el evaluator multiclass.
    classifier_logger = logging.getLogger("mlflow.models.evaluation.evaluators.classifier")
    previous_classifier_level = classifier_logger.level
    classifier_logger.setLevel(logging.ERROR)  # MLflow 3.0 interpreta los labels enteros como desconocidos aunque label_list sea explícita.
    try:
        with mlflow.start_span(name="iris.mlflow_evaluate", span_type="CHAIN", attributes={"model_type": MODEL_TYPE, "test_rows": len(x_test)}) as span:
            evaluation_result = evaluate(
            model=model_info.model_uri,  # URI oficial retornada por log_model.
            model_id=model_info.model_id,  # Identificador del Logged Model en MLflow 3.
            data=evaluation_data,  # Features y target de evaluación.
            targets=config.target_column,  # Nombre parametrizado de la etiqueta.
            model_type="classifier",  # Tipo de problema para el evaluator.
            evaluator_config={"log_model_explainability": False, "label_list": list(range(len(dataset.classes)))},  # Evaluación multiclass explícita.
        )
            span.set_outputs({"metrics": evaluation_result.metrics})
    finally:
        classifier_logger.setLevel(previous_classifier_level)
    evaluator_metric_names = {"accuracy_score", "example_count", "f1_score", "log_loss", "precision_score", "recall_score", "roc_auc"}
    mlflow.log_metrics({f"mlflow_eval_{name}": float(value) for name, value in evaluation_result.metrics.items() if name in evaluator_metric_names and isinstance(value, (int, float))})
    mlflow.set_tag("evaluation_status", "completed")

client = build_registry_client(config.registry_uri)  # Cliente explícito del registry UC.
registered_model = mlflow.register_model(
    model_uri=model_info.model_uri,  # URI oficial retornada por log_model.
    name=config.registered_model_name,  # Modelo UC de tres niveles.
)
registered_version = registered_model.version
deadline = time.time() + config.model_registration_timeout_seconds
while True:
    version_info = client.get_model_version(config.registered_model_name, registered_version)
    if version_info.status == "READY":
        break
    if time.time() >= deadline:
        raise TimeoutError(f"La versión {registered_version} no alcanzó READY.")
    time.sleep(config.model_registration_poll_seconds)
for tag_key, tag_value in {"model_type": MODEL_TYPE, "algorithm": MODEL_TYPE, "framework": MODEL_FRAMEWORK, "stage": "Challenger", "dataset": "iris", "training_type": "baseline"}.items():
    client.set_model_version_tag(config.registered_model_name, registered_version, tag_key, tag_value)
identity = {"model_type": MODEL_TYPE, "framework": MODEL_FRAMEWORK, "stage": "Challenger", "registered_model_name": config.registered_model_name, "registered_model_version": str(registered_version), "challenger_alias": config.challenger_alias, "champion_alias": config.champion_alias, "run_id": run_id}
try:
    client.set_registered_model_alias(
        name=config.registered_model_name,
        alias=config.challenger_alias,
        version=registered_version,
    )
    challenger = client.get_model_version_by_alias(config.registered_model_name, config.challenger_alias)
    if str(challenger.version) != str(registered_version):
        raise RuntimeError(f"El alias {config.challenger_alias} apunta a {challenger.version}, no a {registered_version}.")
except Exception as error:
    with mlflow.start_run(run_id=run_id):
        mlflow.set_tag("challenger_alias_status", "failed")
    raise RuntimeError(f"Falló la asignación de Challenger; run_id={run_id}, version={registered_version}.") from error
with mlflow.start_run(run_id=run_id):
    mlflow.set_tag("challenger_alias_status", "verified")
    mlflow.log_dict(identity, "metadata/model_identity.json")
try:
    registry_evidence = synchronize_model_registry_metadata(
        client, config=config, version=registered_version, run_id=run_id
    )
except Exception as error:
    with mlflow.start_run(run_id=run_id):
        mlflow.set_tag("registry_metadata_status", "failed")
    raise RuntimeError("Falló la sincronización de metadata del registry.") from error
with mlflow.start_run(run_id=run_id):
    mlflow.set_tag("registry_metadata_status", "verified")
    mlflow.log_dict(registry_evidence, "metadata/model_registry_verification.json")
try:
    champion_version = client.get_model_version_by_alias(config.registered_model_name, config.champion_alias).version
except MlflowException as error:
    error_message = str(error).lower()
    missing_alias = any(marker in error_message for marker in ("does not exist", "not found", "resource_does_not_exist"))
    if not missing_alias:
        raise
    champion_version = None
print({"run_id": run_id, "model_type": MODEL_TYPE, "framework": MODEL_FRAMEWORK, "registered_version": registered_version, "challenger_alias": config.challenger_alias, "champion_version": champion_version})

## 5. Verificar que el entrenamiento terminó correctamente

La ejecución sólo se considera completa si `MlflowClient` encuentra parámetros, métricas y los artefactos mínimos bajo `evaluation/`. Luego el modelo recién registrado se carga con `mlflow.pyfunc.load_model` y se prueban tres predicciones.

Al finalizar, guarda `Run ID`, `Model ID`, `Model URI` y `Registered version`. Si la última celda termina sin excepción y muestra predicciones, el notebook concluyó correctamente. Revisa el run en MLflow y confirma que `Challenger` apunta a esa versión.

In [ ]:
client = mlflow.MlflowClient()
logged_run = client.get_run(run_id)
if not logged_run.data.params or not logged_run.data.metrics:
    raise RuntimeError(f"El run {run_id} no contiene tracking completo.")
artifact_paths = {artifact.path for artifact in client.list_artifacts(run_id, "evaluation")}
required = {"evaluation/metrics_summary.json", "evaluation/classification_by_class.json", "evaluation/classification_report.json", "evaluation/confusion_matrix.png"}
if not required.issubset(artifact_paths):
    raise RuntimeError(f"Faltan artefactos de evaluación en {run_id}: {required - artifact_paths}")
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)
print(f"Experiment ID: {run.info.experiment_id}")
print(f"Run ID: {run_id}")
print(f"Model ID: {model_info.model_id}")
print(f"Model URI: {model_info.model_uri}")
print(f"Registered version: {registered_version}")
print(f"Trace ID: {mlflow.get_last_active_trace_id()}")
print(f"Evaluation metrics: {evaluation_result.metrics}")
print(f"Sample predictions: {loaded_model.predict(x_test.head(3)).tolist()}")